# Run InSituCNV

Infer copy-number variation (CNV) profiles from an image-based spatial
transcriptomics dataset, step by step, using the `insitucnv` package API.

**What you need**

- an `.h5ad` with raw counts (in `layers['raw_counts']` or `X`),
  `obsm['spatial']`, and an `obs` column of cell-type / region labels that
  identifies non-tumor *reference* cells;
- gene symbols in `var_names` (human, GRCh38). Pass `gene_reference_path=` a
  CSV with `gene_name,chromosome,start,end` if your annotation differs.

A neighbor graph is built automatically if the file does not already have one.
Run this notebook from the repo root, or set `DATA_PATH` to your file.

In [ ]:
from pathlib import Path

import scanpy as sc

import insitucnv as icv

sc.settings.verbosity = 1

## 1. Settings

Edit these for your dataset. Leaving `DATA_PATH` at its default downloads a
small example dataset so the notebook runs end to end.

In [ ]:
SAMPLE_NAME = 'example'
DATA_PATH = None  # e.g. Path('data/your_dataset.h5ad'); None -> download the example
OUTPUT_DIR = Path('results') / SAMPLE_NAME

RAW_LAYER = 'raw_counts'
SPATIAL_KEY = 'spatial'
REFERENCE_KEY = 'cell_type'          # obs column with cell-type / region labels
REFERENCE_CATEGORIES = ['T_cells', 'B_cells', 'Stromal']  # non-tumor reference labels

CLUSTER_RESOLUTIONS = [0.1, 0.2, 0.3]
GENE_REFERENCE_PATH = None           # CSV with gene_name,chromosome,start,end (optional)

In [ ]:
USING_EXAMPLE = DATA_PATH is None
if USING_EXAMPLE:
    DATA_PATH = icv.download_example_dataset()
    REFERENCE_KEY = 'cell_type'

adata = sc.read_h5ad(DATA_PATH)
adata

## 2. Check the input

In [ ]:
if SPATIAL_KEY not in adata.obsm:
    raise KeyError(f"adata.obsm['{SPATIAL_KEY}'] is required for spatial plots")
if REFERENCE_KEY not in adata.obs:
    raise KeyError(f"adata.obs['{REFERENCE_KEY}'] is required as the reference column")
if RAW_LAYER not in adata.layers:
    print(f"copying adata.X into layers['{RAW_LAYER}']")
    adata.layers[RAW_LAYER] = adata.X.copy()

present = [c for c in REFERENCE_CATEGORIES if c in set(adata.obs[REFERENCE_KEY].astype(str))]
if not present:
    present = None  # let InSituCNV choose (it warns which labels it picks)
print('reference categories:', present)
adata.obs[REFERENCE_KEY].value_counts()

In [ ]:
icv.pl.plot_spatial(adata, color=REFERENCE_KEY, spatial_key=SPATIAL_KEY, show=True)

## 3. Preprocess and run inferCNV

`prepare_cnv_input` restores raw counts, normalizes, builds the smoothing
graph if needed, smooths over it, log-normalizes and adds genomic positions.

In [ ]:
adata = icv.tl.prepare_cnv_input(
    adata,
    raw_layer=RAW_LAYER,
    gene_reference_path=GENE_REFERENCE_PATH,
)
adata.var[['chromosome', 'start', 'end']].head()

In [ ]:
from insitucnv.workflow import resolve_reference_categories

reference_categories = present or resolve_reference_categories(adata, REFERENCE_KEY)
icv.tl.run_infercnv(adata, reference_key=REFERENCE_KEY, reference_categories=reference_categories)

## 4. Cluster CNV profiles

In [ ]:
icv.tl.compute_cnv_neighbors(adata)
cluster_keys = icv.tl.cluster_cnv_resolutions(adata, resolutions=CLUSTER_RESOLUTIONS, dendrogram=True)
cluster_keys

In [ ]:
# Optional: score each resolution (silhouette / stability / spatial cohesion)
from insitucnv.analysis import find_optimal_clustering

metrics = find_optimal_clustering(adata, resolutions=CLUSTER_RESOLUTIONS, spatial_key=SPATIAL_KEY)
metrics

In [ ]:
PRIMARY_RESOLUTION = CLUSTER_RESOLUTIONS[0]
CLUSTER_KEY = icv.tl.cnv_leiden_key(PRIMARY_RESOLUTION)
adata.obs[CLUSTER_KEY].value_counts()

## 5. Review

Inspect the chromosome heatmap and the spatial plot before annotating.

In [ ]:
icv.pl.plot_chromosome_heatmap(adata, groupby=CLUSTER_KEY, show=True)

In [ ]:
icv.pl.plot_spatial(adata, color=CLUSTER_KEY, spatial_key=SPATIAL_KEY, show=True)

## 6. Annotate tumor / normal clusters

Fill in the cluster labels you read off the heatmap. Nothing is inferred
automatically.

In [ ]:
NORMAL_CLUSTERS = []        # e.g. ['0']
TUMOR_CLUSTERS = []         # optional; leave empty to treat all non-normal as tumor
TUMOR_CLONE_CLUSTERS = []   # tumor clusters with distinct CNV profiles to report as clones

if not NORMAL_CLUSTERS and not TUMOR_CLUSTERS and USING_EXAMPLE:
    # example only: call the clusters that are mostly reference cells 'normal'
    import pandas as pd
    ref = adata.obs[REFERENCE_KEY].astype(str).isin(reference_categories)
    frac = ref.groupby(adata.obs[CLUSTER_KEY], observed=True).mean()
    NORMAL_CLUSTERS = frac.index[frac > 0.5].astype(str).tolist()
    print('example NORMAL_CLUSTERS =', NORMAL_CLUSTERS)

if not NORMAL_CLUSTERS and not TUMOR_CLUSTERS:
    raise ValueError('Edit NORMAL_CLUSTERS and/or TUMOR_CLUSTERS above.')

icv.tl.assign_cnv_status(
    adata, cluster_key=CLUSTER_KEY,
    normal_clusters=NORMAL_CLUSTERS or None,
    tumor_clusters=TUMOR_CLUSTERS or None,
    output_key='cnv_status',
)
if TUMOR_CLONE_CLUSTERS:
    icv.tl.assign_cnv_subclones(
        adata, cluster_key=CLUSTER_KEY,
        tumor_clusters=TUMOR_CLONE_CLUSTERS,
        normal_clusters=NORMAL_CLUSTERS or None,
        output_key='cnv_clone',
    )
adata.obs['cnv_status'].value_counts()

## 7. Save outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(OUTPUT_DIR / 'cluster_resolution_metrics.csv', index=False)
icv.tl.export_cell_groups(adata, OUTPUT_DIR / 'cnv_status_groups.csv', group_key='cnv_status')
if 'cnv_clone' in adata.obs:
    icv.tl.export_cell_groups(adata, OUTPUT_DIR / 'cnv_clone_groups.csv', group_key='cnv_clone')
if 'tumor' in set(adata.obs['cnv_status'].astype(str)):
    icv.tl.export_mean_cnv_per_gene(adata, OUTPUT_DIR / 'mean_cnv_per_gene.tsv')

adata.write(OUTPUT_DIR / f'{SAMPLE_NAME}_InSituCNV.h5ad', compression='gzip')
print('wrote', OUTPUT_DIR)